In [1]:
pip install modelscope

Looking in indexes: https://mirrors.cloud.aliyuncs.com/pypi/simple

[notice] A new release of pip is available: 23.3.2 -> 25.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
from modelscope import snapshot_download
model_dir = snapshot_download("Qwen/Qwen2.5-0.5B-Instruct")


2026-01-09 13:33:19,286 - modelscope - INFO - Target directory already exists, skipping creation.


In [3]:
import torch
print(torch.__version__)

2.9.1+cu128


In [5]:
pip install PyPDF2

Looking in indexes: https://mirrors.cloud.aliyuncs.com/pypi/simple
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 20.7 MB/s eta 0:00:00

[notice] A new release of pip is available: 23.3.2 -> 25.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [9]:
import PyPDF2

with open("人工智能通识教程.pdf", "rb") as f:
    reader = PyPDF2.PdfReader(f)
    text = ""
    for page in reader.pages[:10]:  # 只看前3页
        text += page.extract_text()

print(repr(text[:500]))  # 用 repr 显示 \n \t 等控制符

'人工智能通识教程\n北\u3000京主\u3000编\u3000程国安\u3000刘隆吉\u3000蔡兴壮\n副主编\u3000盖延亮\u3000王家波\u3000高等院校产教融合创新应用系列内 容 简 介\n本书主要介绍人工智能的核心技术与应用发展，内容涵盖人工智能概述、人工智能技术原理、人工智\n能交叉学科及融合技术、人工智能应用方向、人工智能应用场景、人工智能发展展望和人工智能时代的职\n业展望。引入丰富的实践案例和拓展阅读，旨在帮助学生深入理解人工智能的技术脉络与社会影响，掌握相关领域的基础知识，提升技术应用能力，适应数字化时代的发展需求。\n本书结构清晰，理论与实践结合，可作为职业本科院校、高职高专院校人工智能通识课程的教材，也\n可作为对人工智能技术感兴趣的读者的参考资料。\n本书封面贴有清华大学出版社防伪标签，无标签者不得销售。\n版权所有，侵权必究。举报：010-62782989， beiqinquan@tup.tsinghua.edu.cn\n 。\n图书在版编目 (CIP)数据\n人工智能通识教程 / 程国安 , 刘隆吉 , 蔡兴壮主编 .\n北京 : 清华大学出版社 , 2025. 8. -- ( 高等院校产教融合\n创新应用系列 ). -- ISBN 978-7-30'


/usr/local/lib/python3.11/site-packages/PyPDF2/_cmap.py:142: PdfReadWarning: Advanced encoding /UniGB-UTF16-H not implemented yet
  warnings.warn(


In [10]:
# split_ai_tutorial.py
import re
import json
from pathlib import Path
try:
    import PyPDF2
except ImportError:
    raise ImportError("请先安装 PyPDF2: pip install PyPDF2")

def extract_and_split_ai_tutorial(pdf_path="人工智能通识教程.pdf"):
    # === 第一步：读取整个 PDF 文本 ===
    pdf_file = Path(pdf_path)
    if not pdf_file.exists():
        raise FileNotFoundError(f"未找到文件: {pdf_file.absolute()}")

    with open(pdf_file, "rb") as f:
        reader = PyPDF2.PdfReader(f)
        full_text = ""
        for page in reader.pages:
            text = page.extract_text()
            if text:
                full_text += text + "\n"

    print(f"✅ 成功读取 PDF，总字符数: {len(full_text)}")

    # === 第二步：定位正文起始位置 ===
    # 尝试匹配第一章的多种可能写法
    patterns = [
        r'第\s*1\s*章\s*[^\n]{0,20}人工智能',
        r'1\s*[\.．]\s*1\s*[^\n]{0,20}[Aa]rtificial',
        r'1\s*[\.．]\s*1\s*[^\n]{0,20}概述',
        r'第\s*一\s*章',
        r'1\s*[\.．]\s*1'
    ]
    
    main_start = 0
    for pattern in patterns:
        match = re.search(pattern, full_text)
        if match:
            main_start = match.start()
            print(f"🔍 找到正文起始位置 (匹配 '{match.group()}')")
            break
    else:
        print("⚠️ 未识别到第一章，将使用全文（可能包含前言/目录）")
    
    main_text = full_text[main_start:]

    # === 第三步：按标题切分 ===
    # 支持: 1.1 / 1．1 / 2.3.1 / 第3章 / 第三章 等
    title_pattern = (
        r'('
        r'\d+[\.．]\d+(?:[\.．]\d+)*\s+[^。\n]{3,}'          # 1.1, 2.3.1
        r'|第\s*\d+\s*章\s+[^\n]{5,}'                       # 第1章 XXX
        r'|第\s*[一二三四五六七八九十]+\s*章\s+[^\n]{5,}'   # 第一章 XXX
        r')'
    )

    parts = re.split(title_pattern, main_text)
    chunks = []
    
    for i in range(1, len(parts), 2):
        title = parts[i].strip()
        content = parts[i+1].strip() if i+1 < len(parts) else ""
        full_chunk = f"{title}\n{content}".strip()
        
        # 过滤太短或明显非正文的内容
        if len(full_chunk) > 50:
            chunks.append(full_chunk)

    print(f"✂️ 初始切分得到 {len(chunks)} 个块")

    # === 第四步：过滤非知识性章节 ===
    skip_keywords = [
        "习题", "思考题", "练习", "参考文献", "附录", "索引",
        "目录", "前言", "序言", "内容简介", "版权", "防伪",
        "图书在版", "CIP", "主编", "副主编", "ISBN"
    ]
    
    filtered_chunks = []
    for chunk in chunks:
        first_line = chunk.split('\n')[0]
        if not any(kw in first_line for kw in skip_keywords):
            filtered_chunks.append(chunk)
    
    print(f"🧹 过滤后剩余 {len(filtered_chunks)} 个有效知识块")

    # === 第五步：保存结果 ===
    output_file = "chunks_ai_tutorial.json"
    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(filtered_chunks, f, ensure_ascii=False, indent=2)
    
    print(f"💾 已保存至: {Path(output_file).absolute()}")

    # 可选：打印前3个块的标题预览
    print("\n📌 前3个块的标题预览:")
    for i, chunk in enumerate(filtered_chunks[:3]):
        title = chunk.split('\n')[0]
        print(f"  {i+1}. {title}")

    return filtered_chunks

if __name__ == "__main__":
    chunks = extract_and_split_ai_tutorial()

✅ 成功读取 PDF，总字符数: 29962
🔍 找到正文起始位置 (匹配 '第 1 章  人工智能')
✂️ 初始切分得到 88 个块
🧹 过滤后剩余 88 个有效知识块
💾 已保存至: /mnt/workspace/chunks_ai_tutorial.json

📌 前3个块的标题预览:
  1. 第 1 章  人工智能概述 .......................................................................................................................001
  2. 1.2　人工智能的发展 ..................................... 007
  3. 1.4　生成式人工智能 ...................................... 024


In [26]:
import os
os.environ['HF_ENDPOINT'] = 'https://hf-mirror.com'

In [13]:
pip install faiss-cpu

Looking in indexes: https://mirrors.cloud.aliyuncs.com/pypi/simple
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 121.0 MB/s eta 0:00:0000:0100:01

[notice] A new release of pip is available: 23.3.2 -> 25.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [25]:
# 魔搭推荐方式（无需 HF）
from modelscope import snapshot_download
from sentence_transformers import SentenceTransformer
import os

# 1. 从魔搭下载 bge-m3（自动缓存）
model_dir = snapshot_download('AI-ModelScope/bge-m3')

# 2. 用 SentenceTransformer 加载本地路径
model = SentenceTransformer(model_dir)

2026-01-09 14:17:16,021 - modelscope - WARNING - Repo AI-ModelScope/bge-m3 not exists on https://www.modelscope.cn, will try on alternative endpoint https://www.modelscope.ai.
2026-01-09 14:17:16,440 - modelscope - ERROR - Repo AI-ModelScope/bge-m3 not exists on either https://www.modelscope.cn or https://www.modelscope.ai


HTTPError: <Response [404]>

In [38]:
# ai_rag_retriever.py
import os
os.environ['HF_ENDPOINT'] = 'https://hf-mirror.com'
import os
import json
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer
import gradio as gr

# # === 1. 设置 Hugging Face 镜像（必须在导入后、模型加载前）===
# os.environ['HF_ENDPOINT'] = 'https://hf-mirror.com'
# os.environ['HF_HUB_DISABLE_TELEMETRY'] = '1'

class AITutorialRetriever:
    def __init__(self, chunks_path: str = "chunks_ai_tutorial.json", cache_dir: str = "rag_cache"):
        self.chunks_path = chunks_path
        self.cache_dir = cache_dir
        os.makedirs(cache_dir, exist_ok=True)
        
        # === 2. 加载文本块 ===
        if not os.path.exists(chunks_path):
            raise FileNotFoundError(f"❌ 找不到 chunks 文件: {chunks_path}")
        with open(chunks_path, "r", encoding="utf-8") as f:
            self.chunks = json.load(f)
        if not self.chunks:
            raise ValueError("❌ chunks 为空！请先提取并分块 PDF 文本。")
        print(f"📚 成功加载 {len(self.chunks)} 个文本块")

        # === 3. 初始化嵌入模型（自动使用 GPU）===
        print("🧠 加载嵌入模型 BAAI/bge-small-zh-v1.5...")
        self.model = SentenceTransformer('BAAI/bge-small-zh-v1.5')
        print(f"✅ 模型加载完成，运行设备: {self.model.device}")

        # === 4. 尝试从缓存加载向量库 ===
        self.embedding_dim = 512  # bge-small-zh-v1.5 输出维度固定
        self.index_path = os.path.join(cache_dir, "faiss.index")
        self.embeddings_path = os.path.join(cache_dir, "embeddings.npy")

        if os.path.exists(self.index_path) and os.path.exists(self.embeddings_path):
            print("📂 从缓存加载 FAISS 向量库...")
            self.index = faiss.read_index(self.index_path)
            # embeddings 可选加载（检索不需要，但调试可能用）
        else:
            print("⏳ 首次生成嵌入向量（稍等...）")
            embeddings = self.model.encode(
                self.chunks,
                batch_size=128,               # 小模型可加大 batch
                normalize_embeddings=True,
                show_progress_bar=True,
                convert_to_numpy=True
            ).astype('float32')

            # 构建 FAISS 索引
            self.index = faiss.IndexFlatIP(self.embedding_dim)
            self.index.add(embeddings)

            # 保存缓存
            faiss.write_index(self.index, self.index_path)
            np.save(self.embeddings_path, embeddings)
            print(f"💾 向量库已缓存至 {cache_dir}/")

    def retrieve(self, query: str, k: int = 3):
        if not query or not query.strip():
            return []

        # 编码查询（自动使用 GPU）
        query_vec = self.model.encode(
            [query.strip()],
            normalize_embeddings=True,
            convert_to_numpy=True
        ).astype('float32')

        # 检索
        distances, indices = self.index.search(query_vec, min(k, self.index.ntotal))

        results = []
        for i in range(len(indices[0])):
            idx = indices[0][i]
            if idx >= len(self.chunks):
                continue
            text = self.chunks[idx]
            # 标题取第一行或前30字
            title = (text.split('\n')[0] if '\n' in text else text[:30]).strip()
            score = float(distances[0][i])
            results.append({
                "title": title,
                "text": text,
                "score": max(0.0, score)  # 相似度 >= 0
            })
        return sorted(results, key=lambda x: x["score"], reverse=True)
    
def answer_question(query: str):
    if not query.strip():
        return "请输入问题。", ""
    
    results = retriever.retrieve(query, k=3)
    
    # 构建答案（取 top-1 作为主要答案）
    if not results:
        return "未找到相关资料。", ""
    
    main_answer = results[0]["text"]
    # 构建参考来源（带相似度）
    refs = "\n\n".join([
        f"【{i+1}】{res['title']} (相似度: {res['score']:.3f})\n{res['text'][:200]}..."
        for i, res in enumerate(results)
    ])
    
    return main_answer, refs


with gr.Blocks(title="AI 教程问答助手") as demo:
    gr.Markdown("## 🤖 AI 教程智能问答系统")
    gr.Markdown("基于《人工智能导论》文档，使用 `bge-small-zh-v1.5` 向量检索")
    
    with gr.Row():
        with gr.Column(scale=2):
            question = gr.Textbox(label="请输入你的问题", placeholder="例如：人工智能的发展趋势是什么？", lines=2)
            submit_btn = gr.Button("🔍 搜索", variant="primary")
        with gr.Column(scale=3):
            answer = gr.Textbox(label="核心答案", interactive=False, lines=6)
            references = gr.Textbox(label="参考来源（Top 3）", interactive=False, lines=10)
    
    submit_btn.click(
        fn=answer_question,
        inputs=question,
        outputs=[answer, references]
    )
    
    gr.Examples(
        examples=[
            "人工智能的发展趋势是什么？",
            "哪些工作不容易被 AI 取代？",
            "AI 在医疗领域有哪些应用？"
        ],
        inputs=question
    )
# ======================
# 使用示例（你的原始代码完全兼容！）
# ======================
if __name__ == "__main__":
    demo.launch(server_name="0.0.0.0", server_port=7860, share=True)  # share=True 生成公网链接
    

* Running on local URL:  http://0.0.0.0:7860

Could not create share link. Missing file: /root/.cache/huggingface/gradio/frpc/frpc_linux_amd64_v0.3. 

Please check your internet connection. This can happen if your antivirus software blocks the download of this file. You can install manually by following these steps: 

1. Download this file: https://cdn-media.huggingface.co/frpc-gradio-0.3/frpc_linux_amd64
2. Rename the downloaded file to: frpc_linux_amd64_v0.3
3. Move the file to this location: /root/.cache/huggingface/gradio/frpc


ModuleNotFoundError: No module named 'ai_rag_retriever'